# Análise de dados TCP-CII

### Importação dos parâmetros universais

In [1]:
from pathlib import Path
import importlib.util

path = Path("../../../parametros/config.py").resolve()

spec = importlib.util.spec_from_file_location("parametros", path)
parametros = importlib.util.module_from_spec(spec)
spec.loader.exec_module(parametros)

In [2]:
# Parâmetros importados do arquivo config.py
print("Filtrar por quantidade de alelos TCC1:.........................", parametros.filtarar_por_qte_de_alelos_tcc1)
print("Parâmetro de filtragem median binding percentile TCC1:.........", parametros.parametro_de_filtragem_mbp_tcc1)
print("Percentual de match mínimo TCC1:...............................", parametros.percent_match_minimo_tcc1)

Filtrar por quantidade de alelos TCC1:......................... 10
Parâmetro de filtragem median binding percentile TCC1:......... 5
Percentual de match mínimo TCC1:............................... 95.0


In [3]:
import pandas as pd

In [4]:
df = pd.read_csv('./T CELL/DENV 2 - T Cell Prediction - Class I.csv')
df

,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhcpan_el core,netmhcpan_el icore,netmhcpan_el score,netmhcpan_el percentile
0,1,VTRLENLMW,60,68,9,HLA-B*57:01,60,0.01,VTRLENLMW,VTRLENLMW,0.993804,0.01
1,1,ASGKLITEW,303,311,9,HLA-B*57:01,303,0.01,ASGKLITEW,ASGKLITEW,0.990493,0.01
2,1,ASGKLITEW,303,311,9,HLA-B*58:01,303,0.01,ASGKLITEW,ASGKLITEW,0.989009,0.01
3,1,ETAECPNTNR,139,148,10,HLA-A*68:01,483,0.01,ETAEPNTNR,ETAECPNTNR,0.981131,0.01
4,1,QPTELKYSW,107,115,9,HLA-B*53:01,107,0.01,QPTELKYSW,QPTELKYSW,0.980907,0.01
...,...,...,...,...,...,...,...,...,...,...,...,...
36985,1,LKEKEENLVNS,338,348,11,HLA-A*32:01,1025,100.00,KEKENLVNS,KEKEENLVNS,0.000000,100.00
36986,1,LKEKEENLVNS,338,348,11,HLA-B*53:01,1025,100.00,LKEENLVNS,LKEKEENLVNS,0.000000,100.00
36987,1,EKEENLVNSLVT,340,351,12,HLA-A*11:01,1369,100.00,ENLVNSLVT,EKEENLVNSLVT,0.000000,100.00
36988,1,EKEENLVNSLVT,340,351,12,HLA-A*32:01,1369,100.00,EEENLVNSL,EKEENLVNSL,0.000000,100.00


## Selecionando Epítopos por median binding percentile.

In [6]:
df_mbp_m5 = df[df['median binding percentile'] < parametros.parametro_de_filtragem_mbp_tcc1].copy()
print("Filtrando por median binding percentile < ", parametros.parametro_de_filtragem_mbp_tcc1)
df_mbp_m5

Filtrando por median binding percentile <  5


,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhcpan_el core,netmhcpan_el icore,netmhcpan_el score,netmhcpan_el percentile
0,1,VTRLENLMW,60,68,9,HLA-B*57:01,60,0.01,VTRLENLMW,VTRLENLMW,0.993804,0.01
1,1,ASGKLITEW,303,311,9,HLA-B*57:01,303,0.01,ASGKLITEW,ASGKLITEW,0.990493,0.01
2,1,ASGKLITEW,303,311,9,HLA-B*58:01,303,0.01,ASGKLITEW,ASGKLITEW,0.989009,0.01
3,1,ETAECPNTNR,139,148,10,HLA-A*68:01,483,0.01,ETAEPNTNR,ETAECPNTNR,0.981131,0.01
4,1,QPTELKYSW,107,115,9,HLA-B*53:01,107,0.01,QPTELKYSW,QPTELKYSW,0.980907,0.01
...,...,...,...,...,...,...,...,...,...,...,...,...
2519,1,FIEVKNCHW,217,225,9,HLA-A*24:02,217,4.90,FIEVKNCHW,FIEVKNCHW,0.003306,4.90
2520,1,ESEMIIPKNLA,238,248,11,HLA-B*44:02,925,4.90,EEMIIPKNL,ESEMIIPKNL,0.002875,4.90
2521,1,MEIRPLKEKE,333,342,10,HLA-B*44:02,677,4.90,MEIRPLKKE,MEIRPLKEKE,0.002854,4.90
2522,1,FIEVKNCHW,217,225,9,HLA-B*44:02,217,4.90,FIEVKNCHW,FIEVKNCHW,0.002809,4.90


## Agrupando por pepitideos e agregando colunas pertinentes

In [7]:
epitopos_repetidos = (
    df_mbp_m5
    .groupby('peptide', as_index=False)
    .agg(
        start=("start", "first"),
        end=("end", "first"),
        qte_de_alelos=("allele", "nunique"),
        median_binding_percentile=(
            "median binding percentile",
            "median"
        ),
        alelos=(
            "allele",
            lambda x: ", ".join(sorted(x.unique()))
        )
    )
)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,AAIKDNRAV,186,194,7,2.70,"HLA-A*02:03, HLA-A*02:06, HLA-A*68:02, HLA-B*0..."
1,AAIKDNRAVH,186,195,2,4.20,"HLA-A*30:02, HLA-B*15:01"
2,ADMGYWIESAL,196,206,1,3.00,HLA-B*40:01
3,AECPNTNRA,141,149,3,1.00,"HLA-B*40:01, HLA-B*44:02, HLA-B*44:03"
4,AECPNTNRAW,141,150,8,1.70,"HLA-A*01:01, HLA-A*23:01, HLA-B*40:01, HLA-B*4..."
...,...,...,...,...,...,...
585,YSWKTWGKA,113,121,2,4.40,"HLA-A*30:02, HLA-A*68:02"
586,YSWKTWGKAK,113,122,6,2.05,"HLA-A*03:01, HLA-A*11:01, HLA-A*30:01, HLA-A*3..."
587,YSWKTWGKAKM,113,123,2,2.40,"HLA-B*57:01, HLA-B*58:01"
588,YWIESALNDTW,200,210,9,1.10,"HLA-A*23:01, HLA-A*24:02, HLA-A*32:01, HLA-B*4..."


## Filtragem por qte_de_alelos

In [8]:
filtarar_por_qte_de_alelos = 10

In [9]:
# Filtro do número de alelos
epitopos_repetidos = epitopos_repetidos[
    epitopos_repetidos["qte_de_alelos"] >= filtarar_por_qte_de_alelos
].reset_index(drop=True)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,ASGKLITEW,303,311,12,1.600,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2..."
1,AVHADMGYW,193,201,10,1.645,"HLA-A*23:01, HLA-A*26:01, HLA-A*30:02, HLA-A*3..."
2,CHWPKSHTL,223,231,12,2.600,"HLA-A*23:01, HLA-A*24:02, HLA-A*30:02, HLA-A*3..."
3,CTLPPLRYR,316,324,11,0.960,"HLA-A*01:01, HLA-A*03:01, HLA-A*11:01, HLA-A*2..."
4,DTWKIEKASF,208,217,12,2.950,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2..."
5,ELKYSWKTW,110,118,11,1.500,"HLA-A*23:01, HLA-A*24:02, HLA-A*26:01, HLA-A*3..."
6,FIEVKNCHW,217,225,10,3.200,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*3..."
7,FITDNVHTW,20,28,22,1.150,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0..."
8,FQPESPSKL,34,42,18,1.600,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0..."
9,FTTNIWLKL,163,171,16,1.650,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0..."


## Sorting por median_biding_percentile

In [10]:
epitopos_repetidos = (
    epitopos_repetidos
    .sort_values(
        ["median_binding_percentile", "qte_de_alelos"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,IFITDNVHTW,19,28,11,0.340,"HLA-A*23:01, HLA-A*24:02, HLA-A*26:01, HLA-A*3..."
1,SCTLPPLRYR,315,324,10,0.660,"HLA-A*01:01, HLA-A*03:01, HLA-A*11:01, HLA-A*2..."
2,GIFITDNVHTW,18,28,10,0.770,"HLA-A*23:01, HLA-A*24:02, HLA-A*26:01, HLA-A*3..."
3,RSLRPQPTELKY,102,113,12,0.820,"HLA-A*01:01, HLA-A*03:01, HLA-A*11:01, HLA-A*3..."
4,QPTELKYSW,107,115,14,0.835,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2..."
5,CTLPPLRYR,316,324,11,0.960,"HLA-A*01:01, HLA-A*03:01, HLA-A*11:01, HLA-A*2..."
6,RSLRPQPTEL,102,111,14,1.080,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*0..."
7,FITDNVHTW,20,28,22,1.150,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0..."
8,IESALNDTW,202,210,10,1.150,"HLA-A*23:01, HLA-A*24:02, HLA-A*32:01, HLA-B*3..."
9,STESHNQTF,125,133,18,1.200,"HLA-A*01:01, HLA-A*02:06, HLA-A*23:01, HLA-A*2..."


## Separando epítopos e criando arquivo FASTA para IEDB analysis resource

In [11]:
pepitides = epitopos_repetidos.peptide

with open("./peptideos_tcell_1.fasta", "w") as f:
    for i, peptide in enumerate(pepitides, start=1):
        f.write(f">NP {i}\n")
        f.write(f"{peptide}\n")
        
pepitides

0       IFITDNVHTW
1       SCTLPPLRYR
2      GIFITDNVHTW
3     RSLRPQPTELKY
4        QPTELKYSW
5        CTLPPLRYR
6       RSLRPQPTEL
7        FITDNVHTW
8        IESALNDTW
9        STESHNQTF
10       RAVHADMGY
11       SQHNYRPGY
12      RSCTLPPLRY
13     SLRPQPTELKY
14     RPQPTELKYSW
15       RPQPTELKY
16     RSCTLPPLRYR
17       SALNDTWKI
18       HTWTEQYKF
19       ELKYSWKTW
20       NVHTWTEQY
21       FQPESPSKL
22       ASGKLITEW
23       AVHADMGYW
24       FTTNIWLKL
25       ITPELNHIL
26      HTQITGPWHL
27       TQITGPWHL
28     MLSTESHNQTF
29      SLRPQPTELK
30       RSCTLPPLR
31       SCTLPPLRY
32       KLKEKQDVF
33       MWKQITPEL
34       IMQAGKRSL
35     KQITPELNHIL
36     KRSLRPQPTEL
37       RSVTRLENL
38       KQITPELNH
39       KTWGKAKML
40      LSTESHNQTF
41       HWPKSHTLW
42       SLRPQPTEL
43      KFQPESPSKL
44       CHWPKSHTL
45       ITGPWHLGK
46       ILSENEVKL
47      TASGKLITEW
48       QITPELNHI
49      KQITPELNHI
50      DTWKIEKASF
51       SENEVKLTI
52      LRPQ

### Seqkit remove sequências proteicas contendo gaps e *.

In [12]:
# !seqkit grep -s -v -r -p '[-*]' './Fastas/denv1_NS1_proteinas.fa' > DENV1_seq_filter_all.fasta

### Resultado IEDB analysis resource

In [13]:
conservacy_result = pd.read_csv('./ConservancyResult_tcell_1.csv')
conservacy_result

,Epitope #,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,View details
0,1,NP 1,FITDNVHTW,9,97.00% (840/866),88.89%,100.00%,NaN
1,2,NP 2,TQITGPWHL,9,0.81% (7/866),55.56%,100.00%,NaN
2,3,NP 3,HTWTEQYKF,9,99.77% (864/866),88.89%,100.00%,NaN
3,4,NP 4,SLRPQPTEL,9,93.19% (807/866),66.67%,100.00%,NaN
4,5,NP 5,KLKEKQDVF,9,3.35% (29/866),66.67%,100.00%,NaN
5,6,NP 6,RPQPTELKY,9,97.58% (845/866),55.56%,100.00%,NaN
6,7,NP 7,STESHNQTF,9,44.11% (382/866),66.67%,100.00%,NaN
7,8,NP 8,FQPESPSKL,9,99.42% (861/866),88.89%,100.00%,NaN
8,9,NP 9,KTWGKAKML,9,69.98% (606/866),77.78%,100.00%,NaN
9,10,NP 10,SENEVKLTI,9,52.89% (458/866),77.78%,100.00%,NaN


### Merge da coluna qte_de_alelos ao dataframe conservacy_result

In [14]:
# qte_de_alelos
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "qte_de_alelos"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide")).drop(columns=("View details"))

# alelos
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "alelos"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide")).drop(columns=("Epitope #"))

conservacy_result

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,alelos
0,NP 1,FITDNVHTW,9,97.00% (840/866),88.89%,100.00%,22,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0..."
1,NP 2,TQITGPWHL,9,0.81% (7/866),55.56%,100.00%,22,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*2..."
2,NP 3,HTWTEQYKF,9,99.77% (864/866),88.89%,100.00%,21,"HLA-A*01:01, HLA-A*02:06, HLA-A*11:01, HLA-A*2..."
3,NP 4,SLRPQPTEL,9,93.19% (807/866),66.67%,100.00%,21,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0..."
4,NP 5,KLKEKQDVF,9,3.35% (29/866),66.67%,100.00%,20,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0..."
5,NP 6,RPQPTELKY,9,97.58% (845/866),55.56%,100.00%,19,"HLA-A*01:01, HLA-A*03:01, HLA-A*11:01, HLA-A*2..."
6,NP 7,STESHNQTF,9,44.11% (382/866),66.67%,100.00%,18,"HLA-A*01:01, HLA-A*02:06, HLA-A*23:01, HLA-A*2..."
7,NP 8,FQPESPSKL,9,99.42% (861/866),88.89%,100.00%,18,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0..."
8,NP 9,KTWGKAKML,9,69.98% (606/866),77.78%,100.00%,18,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*0..."
9,NP 10,SENEVKLTI,9,52.89% (458/866),77.78%,100.00%,18,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:06, HLA-A*2..."


### Gerando a coluna percent_match para filtrar os epitopos com percentagem de match maior que 50%

In [15]:
col = "Percent of protein sequence matches at identity <= 100%"

conservacy_result["percent_match"] = (
    conservacy_result[col]
    .astype(str)
    .str.extract(r"(\d+(?:\.\d+)?)")[0]
    .astype(float)
)

conservacy_result

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,alelos,percent_match
0,NP 1,FITDNVHTW,9,97.00% (840/866),88.89%,100.00%,22,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0...",97.00
1,NP 2,TQITGPWHL,9,0.81% (7/866),55.56%,100.00%,22,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*2...",0.81
2,NP 3,HTWTEQYKF,9,99.77% (864/866),88.89%,100.00%,21,"HLA-A*01:01, HLA-A*02:06, HLA-A*11:01, HLA-A*2...",99.77
3,NP 4,SLRPQPTEL,9,93.19% (807/866),66.67%,100.00%,21,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0...",93.19
4,NP 5,KLKEKQDVF,9,3.35% (29/866),66.67%,100.00%,20,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0...",3.35
5,NP 6,RPQPTELKY,9,97.58% (845/866),55.56%,100.00%,19,"HLA-A*01:01, HLA-A*03:01, HLA-A*11:01, HLA-A*2...",97.58
6,NP 7,STESHNQTF,9,44.11% (382/866),66.67%,100.00%,18,"HLA-A*01:01, HLA-A*02:06, HLA-A*23:01, HLA-A*2...",44.11
7,NP 8,FQPESPSKL,9,99.42% (861/866),88.89%,100.00%,18,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0...",99.42
8,NP 9,KTWGKAKML,9,69.98% (606/866),77.78%,100.00%,18,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*0...",69.98
9,NP 10,SENEVKLTI,9,52.89% (458/866),77.78%,100.00%,18,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:06, HLA-A*2...",52.89


### Sort e filtragem por percent_match e presença em alelos

In [16]:
# Parametros
percent_match_minimo = 95.0

In [17]:
# Filtro do Percent match
conservacy_result_filtered = (
    conservacy_result[conservacy_result["percent_match"] >= percent_match_minimo]
    .sort_values(
            by="percent_match", 
            ascending=False
        )
    ).reset_index(drop=True)

conservacy_result_filtered

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,alelos,percent_match
0,NP 53,RSCTLPPLR,9,99.88% (865/866),88.89%,100.00%,10,"HLA-A*03:01, HLA-A*11:01, HLA-A*30:01, HLA-A*3...",99.88
1,NP 14,NVHTWTEQY,9,99.88% (865/866),88.89%,100.00%,16,"HLA-A*01:01, HLA-A*03:01, HLA-A*11:01, HLA-A*2...",99.88
2,NP 3,HTWTEQYKF,9,99.77% (864/866),88.89%,100.00%,21,"HLA-A*01:01, HLA-A*02:06, HLA-A*11:01, HLA-A*2...",99.77
3,NP 36,RSCTLPPLRY,10,99.77% (864/866),90.00%,100.00%,11,"HLA-A*01:01, HLA-A*03:01, HLA-A*11:01, HLA-A*2...",99.77
4,NP 47,VEDYGFGVF,9,99.77% (864/866),88.89%,100.00%,11,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2...",99.77
5,NP 22,SCTLPPLRY,9,99.77% (864/866),88.89%,100.00%,13,"HLA-A*01:01, HLA-A*03:01, HLA-A*11:01, HLA-A*2...",99.77
6,NP 8,FQPESPSKL,9,99.42% (861/866),88.89%,100.00%,18,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0...",99.42
7,NP 45,TASGKLITEW,10,99.42% (861/866),90.00%,100.00%,11,"HLA-A*23:01, HLA-A*24:02, HLA-A*26:01, HLA-A*3...",99.42
8,NP 27,ASGKLITEW,9,99.42% (861/866),88.89%,100.00%,12,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2...",99.42
9,NP 60,SVTRLENLM,9,99.19% (859/866),55.56%,100.00%,10,"HLA-A*02:06, HLA-A*26:01, HLA-A*30:02, HLA-A*3...",99.19
